## Setup (one time)

Run the installer once from a terminal in the repo root (it needs conda):

```bash
bash install.sh
```

It creates the `agentp_env` conda env and puts `plink2`, `bgenix` and `regenie` in `~/.local/bin`. Then select the **agentp_env** kernel for this notebook.

## Running the example

### Point this kernel at the agentp_env tools

In [ ]:
import os, sys, shutil, platform

# agentp shells out to `python`, plink2, bgenix and regenie. Put agentp_env and
# ~/.local/bin first on PATH so they resolve correctly however Jupyter was launched.
env_bin = os.path.dirname(sys.executable)
local_bin = os.path.expanduser("~/.local/bin")
os.environ["PATH"] = os.pathsep.join([env_bin, local_bin, os.environ["PATH"]])

print("Python:", sys.executable)
print("Platform:", platform.platform())
for tool in ["python", "plink2", "bgenix", "regenie"]:
    path = shutil.which(tool)
    print(f"{tool:8} -> {path}")
    assert path, f"{tool} not found - run install.sh first"

### Import agentp and create the Project

In [ ]:
import warnings
warnings.filterwarnings("ignore", message="pkg_resources is deprecated")

from agentp import Project, SummaryTWAS, IndividualTWAS, GWAS

# Paths are relative to the repo root, where this notebook lives
example_dir = os.path.abspath("example")
data_dir = os.path.join(example_dir, "example_data")
assert os.path.isdir(data_dir), f"example data not found at {data_dir}"

# Steps whose outputs already exist in the project folder are skipped.
# Use a new folder name to rerun everything from scratch.
project_dir = os.path.join(example_dir, "example_project_rerun")
project = Project(project_dir)

### Summary TWAS (JTI hippocampus) + add GWAS

In [ ]:
stwas = SummaryTWAS(project, "JTI", "hippocampus")

stwas.add_gwas(
    "amygdala_volume",
    os.path.join(data_dir, "vol_mean_amygdala.regenie")
)

### Run summary TWAS

In [ ]:
stwas.run_twas("amygdala_volume")

### Individual TWAS

In [ ]:
gtwas = IndividualTWAS(project, "FUS", "caudate")

### Add subject/covariate/phenotype/genotype files

In [ ]:
project.set_subjects(os.path.join(data_dir, "subjects.txt"))
project.add_covariates(os.path.join(data_dir, "covariates.csv"))
project.add_phenotypes(os.path.join(data_dir, "volumes.csv"))

project.add_genotypes(os.path.join(data_dir, "genotypes"), "test_c*.bgen")

### Predict GReX

In [ ]:
gtwas.predict_grex()

### Run individual TWAS

In [ ]:
gtwas.run_twas("putamen_volume")

### Run GWAS on hippocampus_volume

In [ ]:
gwas = GWAS(project)
gwas.run("hippocampus_volume")

### Results

In [ ]:
import pandas as pd

results = {
    "sTWAS (JTI hippocampus, amygdala_volume)": "sTWAS/JTI_hippocampus/amygdala_volume.csv",
    "iTWAS (FUS caudate, putamen_volume)": "iTWAS/FUS_caudate/putamen_volume.csv",
}
for label, rel in results.items():
    df = pd.read_csv(os.path.join(project.path, rel), sep="\t")
    print(f"{label}: {len(df)} genes")
    display(df.sort_values("pvalue").head())

print("GWAS:", os.path.join(project.path, "GWAS", "hippocampus_volume.regenie"))